# GeoJSON Quickstart for Hanoi

This notebook shows how the Hanoi boundary GeoJSON is loaded, what a GeoJSON feature looks like, and how listing points overlay on top of that region.

## 1. Imports and setup
We add `src/` to `sys.path` so the notebook can reuse the project modules directly.

In [ ]:
from pathlib import Path
import json
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.append(str(PROJECT_ROOT / 'src'))

from hanoi_real_estate.gis import (
    HANOI_BOUNDARY_PATH,
    build_listing_geodataframe,
    load_hanoi_boundary,
    load_hanoi_boundary_geojson,
)

plt.style.use('seaborn-v0_8-whitegrid')

## 2. Load the Hanoi boundary
On first run, the helper may download the Hanoi polygon from OpenStreetMap via OSMnx and cache it locally as GeoJSON.

In [ ]:
boundary_gdf = load_hanoi_boundary()
boundary_gdf[['display_name', 'geometry']]

In [ ]:
HANOI_BOUNDARY_PATH

## 3. Inspect the raw GeoJSON structure
A GeoJSON file is usually a `FeatureCollection` with one or more `features`, and each feature contains `properties` plus a `geometry`.

In [ ]:
boundary_geojson = load_hanoi_boundary_geojson()
list(boundary_geojson.keys())

In [ ]:
len(boundary_geojson['features'])

In [ ]:
first_feature = boundary_geojson['features'][0]
{
    'type': first_feature['type'],
    'property_keys': list(first_feature['properties'].keys())[:10],
    'geometry_type': first_feature['geometry']['type'],
}

In [ ]:
print(json.dumps(first_feature, indent=2)[:2000])

## 4. Load listing points as a GeoDataFrame
`GeoDataFrame` is just a normal DataFrame with a geometry column, which makes spatial operations much easier.

In [ ]:
listing_gdf = build_listing_geodataframe(active_only=True)
listing_gdf[['Mã tin', 'Tiêu đề', 'Huyện', 'Latitude', 'Longitude', 'geometry']].head()

In [ ]:
listing_gdf.crs

## 5. Overlay listing points on the Hanoi boundary
This is the basic visual pattern we will reuse in the dashboard with pydeck.

In [ ]:
sample_points = listing_gdf.sample(n=min(600, len(listing_gdf)), random_state=42) if not listing_gdf.empty else listing_gdf
fig, ax = plt.subplots(figsize=(10, 10))
boundary_gdf.plot(ax=ax, color='#d6eaf8', edgecolor='#2471a3', linewidth=1.2)
sample_points.plot(ax=ax, color='#c0392b', markersize=6, alpha=0.45)
ax.set_title('Sample Hanoi Listings over Hanoi Boundary')
ax.set_axis_off()
plt.show()

## 6. Quick spatial filter example
This checks whether each listing point falls inside the Hanoi polygon.

In [ ]:
joined = gpd.sjoin(listing_gdf, boundary_gdf[['display_name', 'geometry']], how='left', predicate='within')
joined['inside_hanoi'] = joined['display_name'].notna()
joined['inside_hanoi'].value_counts(dropna=False)

In [ ]:
joined.loc[~joined['inside_hanoi'], ['Mã tin', 'Tiêu đề', 'Địa chỉ', 'Latitude', 'Longitude']].head(20)

## What to notice
- GeoJSON stores shapes as coordinates inside `features`.
- GeoPandas turns those shapes into geometry objects.
- Spatial joins let us compare listing points against regions for QA before building heatmaps or routing.